In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import polars as pl
import matplotlib.pyplot as plt

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "scripts"))

from conexion import run_query

In [ ]:
metro_ecom = pl.read_parquet("fuga_input_metro_ecommerce.parquet")

In [ ]:
#cosntrucción rfm
#primero se define la fecha de corte
metro_ecom = metro_ecom.with_columns(pl.col("fecha_pedido").str.to_date("%Y%m%d").alias("fecha"))
metro_ecom

In [ ]:
#veamos la actividad del cliente, si la historia es corta y muchos han comprado solo 1 vez se justificaría que más del 50% hayan comprado 1 vez y no volvieron
antiguedad_cliente = (
metro_ecom.group_by("dni").agg(
[pl.col("fecha").min().alias("primera_compra"),pl.col("fecha").max().alias("ultima_compra"),])
.with_columns((pl.col("ultima_compra") - pl.col("primera_compra")).dt.total_days().alias("antiguedad_dias"))
)

print("clientes que compraron 1 sola vez: ", (antiguedad_cliente.filter(pl.col("antiguedad_dias")==0)).shape[0])

In [ ]:
plt.hist(antiguedad_cliente["antiguedad_dias"], bins=50)
plt.xlabel("Antigüedad (días)")
plt.show()

> [Conclusion]: Más de la mitad de clientes han comprado 1 sola vez en ecommerce, descartar clientes con historial de compra de 1 sola vez antes de modelar su historial.
>
> [Corrección]: esto aplicaba al `rfm`, donde `frecuencia` era el conteo crudo y empezaba en 1. En el `rfmt` la definición cambia a `n_compras - 1`, así que esos clientes pasan a valer 0 y **sí se conservan**. Ver la comprobación al final del notebook.

In [ ]:
# mejora y calculo de rfmt
# T: el número de períodos de tiempo transcurridos desde la primera compra del cliente.
fecha_corte = metro_ecom["fecha"].max()
rfmt = (metro_ecom.group_by("sk_cliente").agg(
        pl.col("fecha").min().alias("primera_compra"),
        pl.col("fecha").max().alias("ultima_compra"),
        pl.col("fecha_pedido").n_unique().alias("n_compras"),
        pl.col("venta_bruta").sum().alias("monto_total"),
    ).with_columns(
            (pl.col("ultima_compra") - pl.col("primera_compra")).dt.total_days().alias("recencia"),
            (fecha_corte - pl.col("primera_compra")).dt.total_days().alias("T"), #
            (pl.col("n_compras")-1).alias("frecuencia"),
            ).with_columns(
                    pl.when(pl.col("frecuencia") > 0)
                    .then(pl.col("monto_total") / pl.col("n_compras"))  # promedio por evento de compra
                    .otherwise(0.0)
                    .alias("monetary_value")
                ))

In [ ]:
rfmt

## Distribución binomial negativa sobre el RFMT

In [ ]:
import numpy as np
from scipy.stats import nbinom
from scipy.optimize import minimize

x = rfmt["frecuencia"].to_numpy().astype(float)
T = rfmt["T"].to_numpy().astype(float)

media = x.mean()
varianza = x.var()

print(f"media    = {media:.3f}")
print(f"varianza = {varianza:.3f}")
print(f"ratio    = {varianza / media:.1f}")

In [ ]:
# valores iniciales por momentos
r0 = media**2 / (varianza - media)
alpha0 = r0 * T.mean() / media

print(f"r0     = {r0:.4f}")
print(f"alpha0 = {alpha0:.2f}")

In [ ]:
# cada cliente tiene su propio p segun su antiguedad T
def nll(par):
    r, alpha = np.exp(par)
    p = alpha / (alpha + T)
    return -nbinom.logpmf(x, r, p).sum()

res = minimize(nll, np.log([r0, alpha0]), method="Nelder-Mead",
               options={"maxiter": 5000, "fatol": 1e-8, "xatol": 1e-8})

r, alpha = np.exp(res.x)

print(f"r      = {r:.4f}")
print(f"alpha  = {alpha:.2f}")
print(f"lambda = {r / alpha * 365:.2f} compras al año")

In [ ]:
k = np.arange(0, 16)

observado = np.bincount(x.astype(int))[:len(k)] / len(x)

p = alpha / (alpha + T)
ajustado = np.array([nbinom.pmf(ki, r, p).mean() for ki in k])

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(k - 0.2, observado, width=0.4, label="Observado")
plt.bar(k + 0.2, ajustado, width=0.4, label="Binomial negativa")
plt.xlabel("Número de recompras")
plt.ylabel("Proporción de clientes")
plt.title("Distribución observada vs binomial negativa ajustada")
plt.xticks(k)
plt.legend()
plt.show()

### ¿Se deben quitar los clientes con 0 recompras?

No. Con `frecuencia = n_compras - 1`, quien compró 1 vez tiene **0 recompras**: es un dato observado, no un faltante. Además la masa en 0 es la que identifica `r`.

Comprobación ajustando con y sin ellos:

In [ ]:
def ajusta(x_, T_):
    def f(par):
        r_, a_ = np.exp(par)
        return -nbinom.logpmf(x_, r_, a_ / (a_ + T_)).sum()
    res = minimize(f, np.log([0.1, 15.]), method="Nelder-Mead",
                   options={"maxiter": 5000, "fatol": 1e-8, "xatol": 1e-8})
    return np.exp(res.x)

r_con, a_con = ajusta(x, T)

m = x > 0
r_sin, a_sin = ajusta(x[m], T[m])

print(f"con ceros (n={len(x):,}): r={r_con:.4f}  alpha={a_con:.2f}  lambda={r_con / a_con * 365:.2f}")
print(f"sin ceros (n={m.sum():,}): r={r_sin:.4f}  alpha={a_sin:.2f}  lambda={r_sin / a_sin * 365:.2f}")

print(f"\nP(x=0) real      = {(x == 0).mean():.4f}")
print(f"P(x=0) con ceros = {nbinom.pmf(0, r_con, a_con / (a_con + T)).mean():.4f}")
print(f"P(x=0) sin ceros = {nbinom.pmf(0, r_sin, a_sin / (a_sin + T)).mean():.4f}")

print(f"\nmedia real      = {x.mean():.3f}")
print(f"media con ceros = {(r_con * T / a_con).mean():.3f}")
print(f"media sin ceros = {(r_sin * T / a_sin).mean():.3f}")

> [Conclusión]: quitarlos duplica lambda (2.19 → 4.52) y hace que el modelo prediga 21% de clientes sin recompra cuando la realidad es 53%. **No se quitan.**
>
> El desajuste que queda en k=1 no se arregla quitándolos: viene de que este modelo asume que nadie abandona. Eso lo corrige el paso siguiente (BG/NBD).